# Prognozowanie rezygnacji klientów z usług telekomunikacyjnych z wykorzystaniem AutoML

---

**Opis problemu:**

Firma telekomunikacyjna chce przewidzieć, którzy klienci z dużym prawdopodobieństwem **zrezygnują z usług w najbliższym miesiącu** (tzw. *churn prediction*), aby skierować do nich działania retencyjne.

Dysponuje danymi historycznymi zawierającymi m.in.:

* dane demograficzne klienta (wiek, płeć, region),
* informacje o typie usługi (abonament, prepaid, internet, TV, pakiety),
* statystyki dotyczące korzystania z usług (liczba minut, transfer, faktury, zgłoszenia),
* zmienną docelową: `churn` (`1` – rezygnacja, `0` – pozostanie klientem).

---

**Cel:**

Zbudować model, który:

* przewiduje prawdopodobieństwo rezygnacji (binary classification),
* automatycznie wybiera najlepsze algorytmy i parametry,
* umożliwia interpretację wyników (np. które cechy najbardziej wpływają na rezygnację),

---

**Wykorzystanie AutoML:**

W projekcie można wykorzystać:

* **H2O AutoML** – dla dużej skali i możliwości integracji,
* **MLJAR-Supervised** – dla szybkiej eksploracji i gotowych raportów,
* **AutoGluon** – jeśli zależy nam na pracy z danymi tabelarycznymi + testowaniu wielu typów modeli.

---

**Rezultaty, jakie mogą być oczekiwane:**

* metryki: `AUC`, `logloss`, `recall` (ważne przy danych niezbalansowanych),
* lista klientów wysokiego ryzyka (z `predict_proba` > 0.8),
* wykresy ważności cech,
* raport do dalszego wykorzystania przez dział marketingu/retencji.

# Dane

**Zbiór danych: Telco Customer Churn**

[https://www.kaggle.com/datasets/blastchar/telco-customer-churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

---

**Informacje o zbiorze**:

* Każdy wiersz = jeden klient.
* Zmienna docelowa: `Churn` (`Yes`/`No`)
* Zawiera dane takie jak:

  * `tenure` – czas korzystania z usług
  * `MonthlyCharges`, `TotalCharges`
  * `Contract`, `PaymentMethod`, `InternetService` itd.
* Zmiennych jest \~20, część kategoryczna, część numeryczna.

---

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

In [2]:
# Wczytanie danych – przykładowy zbiór churn (np. z Kaggle lub własny)
df = pd.read_csv('./Data/customer_churn.csv')

# Wyświetlenie podstawowych informacji
print(df.info())
print(df['Churn'].value_counts())

# Konwersja zmiennej docelowej do 0/1 (jeśli jest tekstowa)
df['churn'] = df['Churn'].map({'No': 0, 'Yes': 1})
df = df.drop(columns=['customerID', 'Churn']) 

# Podział na cechy i etykiety
X = df.drop(columns='churn')
y = df['churn']

# Podział na zbiór treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


**UWAGA**: powyższy kod można zmodyfikować, jeśli ułatwi to zadanie :)

# autoML

In [3]:
# tu wpisz swoje rozwiązanie